<a href="https://colab.research.google.com/github/juliaramosguedes/fiap-fase-1-aurora-siger/blob/main/aurora_siger_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ☄️ Missão Aurora Siger — Relatório Operacional de Pré-Decolagem

**Curso:** Ciência da Computação — FIAP  
**Participantes:** Julia Ramos, Carlos Eugenio, Julio Guma, Matheus Fuchens  
**Fase:** 1 — Atividade Integradora

---

Relatório de pré-decolagem da missão fictícia Aurora Siger. Seções:

1. 🛰 Telemetria
2. 🛸 Verificação de pré-lançamento
3. ⚡ Análise energética
4. 👾 Análise assistida por IA

> 📝 Valores marcados como `# SIMULATED` foram estimados por ordem de grandeza. Ver [`telemetry.md`](telemetry.md).

In [1]:
# =============================================================================
# IMPORTS — all dependencies declared upfront to avoid cell-order issues
# =============================================================================

from __future__ import annotations

from dataclasses import dataclass
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
)

# =============================================================================
# GLOBAL CONSTANTS
# =============================================================================

RANDOM_STATE: int = 42   # fixed seed — makes all models and splits reproducible
REPORT_WIDTH: int = 60   # character width used by every section separator

SEPARATOR: str = "=" * REPORT_WIDTH

np.random.seed(RANDOM_STATE)

# =============================================================================
# DISPLAY HELPERS
# =============================================================================


def format_status(flag: bool) -> str:
    """Return a human-readable status string for a boolean flag."""
    return "OK" if flag else "FALHA"

In [2]:
# =============================================================================
# DEPENDENCY CHECK — reads requirements.txt; run before any other cell
# =============================================================================

import importlib, pathlib, re

# Map: install name → import name (None = skip import check)
_INSTALL_TO_IMPORT = {"scikit-learn": "sklearn", "jupyter": None}


def _parse_requirements(path: str = "requirements.txt") -> list:
    """Parse package names from requirements.txt."""
    req = pathlib.Path(path)
    if not req.exists():
        return []
    return [
        re.split(r"[>=<!~\s]", line.strip())[0]
        for line in req.read_text().splitlines()
        if line.strip() and not line.startswith("#")
    ]


packages    = _parse_requirements()
check_pairs = [(pkg, _INSTALL_TO_IMPORT.get(pkg, pkg)) for pkg in packages]
check_pairs = [(inst, imp) for inst, imp in check_pairs if imp is not None]
missing     = [inst for inst, imp in check_pairs
               if importlib.util.find_spec(imp) is None]

if missing:
    print(f"💥 Módulos ausentes: {missing}")
    print(f"   Executar: pip install {' '.join(missing)}")
    print("   Missão em espera até resolução.")
else:
    found = [imp for _, imp in check_pairs]
    print(f"🚀 Sistema de bordo inicializado. Dependências OK: {found}")
    print("   Tudo verde. Pronto para decolagem.")

🚀 Sistema de bordo inicializado. Dependências OK: []
   Tudo verde. Pronto para decolagem.


---
## 🛰 Seção 1 — Telemetria

Leituras de sensores:
- Temperatura interna e externa (°C)
- Integridade estrutural (flag binário 0/1)
- Nível de energia (%)
- Pressão dos tanques (bar)
- Status dos módulos críticos (booleano por subsistema)

> 📡 Fontes e justificativas detalhadas em [`telemetry.md`](telemetry.md).

In [3]:
# =============================================================================
# SECTION 1 — TELEMETRY DATA
# Sources documented in telemetry.md
# =============================================================================


@dataclass(frozen=True)
class TelemetryReading:
    """Immutable snapshot of spacecraft telemetry at pre-launch verification."""

    # Temperatures (°C) — Source: MIT OCW Satellite Engineering (2003)
    internal_temperature: float   # Electronics bay
    external_temperature: float   # Hull / structural skin

    # Structural integrity flag — Source: ESA Mars Express right_flag model
    structural_integrity: int     # 1 = nominal, 0 = failure

    # Energy level (%) — Source: ESA Advanced Concepts Team (2021)
    energy_level_pct: float

    # Propulsion tank pressure (bar) — Source: NASA SBIR (SIMULATED order of magnitude)
    tank_pressure_bar: float

    # Critical subsystem flags — Source: ESA Mars Express ESOC subsystems
    propulsion_ok: bool
    power_ok: bool
    comms_ok: bool
    thermal_ok: bool
    navigation_ok: bool


# --- Leitura de telemetria simulada ---
TELEMETRY = TelemetryReading(
    internal_temperature=22.5,    # °C — nominal, within -20 to +70°C
    external_temperature=-45.0,   # °C — nominal, within -100 to +125°C
    structural_integrity=1,       # 1 = intact
    energy_level_pct=87.3,        # % — above 60% minimum
    tank_pressure_bar=34.8,       # bar — within 20–40 bar
    propulsion_ok=True,
    power_ok=True,
    comms_ok=True,
    thermal_ok=True,
    navigation_ok=True,
)

# --- Parâmetros do sistema de energia (SIMULATED — ordem de grandeza) ---
TOTAL_CAPACITY_KWH: float    = 120.0  # kWh — medium-class spacecraft reference
LAUNCH_CONSUMPTION_KW: float =   8.5  # kW  — estimated peak draw at launch
SYSTEM_EFFICIENCY: float     =  0.92  # η   — Cap. 7 FIAP; new systems ~92%
LOSS_FACTOR: float           =  0.08  # 8% resistive losses — Cap. 7 FIAP: P = I²×R


def print_telemetry(reading: TelemetryReading) -> None:
    """Print a formatted telemetry snapshot. Display-only, no return value."""
    print(SEPARATOR)
    print("🛰  TELEMETRIA DE PRÉ-DECOLAGEM - AURORA SIGER")
    print("    Sensores a postos. Leituras nominais recebidas.")
    print(SEPARATOR)
    print(f"  Temperatura interna   : {reading.internal_temperature:>8.1f} °C")
    print(f"  Temperatura externa   : {reading.external_temperature:>8.1f} °C")
    print(f"  Integridade estrutural: {reading.structural_integrity:>8}   (1=OK, 0=FALHA)")
    print(f"  Nível de energia      : {reading.energy_level_pct:>8.1f} %")
    print(f"  Pressão dos tanques   : {reading.tank_pressure_bar:>8.1f} bar")
    print(f"  Propulsão             : {format_status(reading.propulsion_ok):>8}")
    print(f"  Gestão de energia     : {format_status(reading.power_ok):>8}")
    print(f"  Comunicações          : {format_status(reading.comms_ok):>8}")
    print(f"  Controle térmico      : {format_status(reading.thermal_ok):>8}")
    print(f"  Navegação / ADCS      : {format_status(reading.navigation_ok):>8}")
    print(SEPARATOR)


print_telemetry(TELEMETRY)

🛰  TELEMETRIA DE PRÉ-DECOLAGEM - AURORA SIGER
    Sensores a postos. Leituras nominais recebidas.
  Temperatura interna   :     22.5 °C
  Temperatura externa   :    -45.0 °C
  Integridade estrutural:        1   (1=OK, 0=FALHA)
  Nível de energia      :     87.3 %
  Pressão dos tanques   :     34.8 bar
  Propulsão             :       OK
  Gestão de energia     :       OK
  Comunicações          :       OK
  Controle térmico      :       OK
  Navegação / ADCS      :       OK


---
## 🛸 Seção 2 — Verificação de Pré-Lançamento

In [4]:
# =============================================================================
# SECTION 2 — VERIFICATION (pure functions · fail-fast)
# =============================================================================
# Safety thresholds are the single source of truth for verification
# and for AI dataset generation in Section 4.
# Sources documented in telemetry.md
# =============================================================================

# --- Safety thresholds ---
# Source: MIT OCW Satellite Engineering (2003); ESA Bulletin 87
TEMP_INTERNAL_MIN: float = -20.0
TEMP_INTERNAL_MAX: float =  70.0
TEMP_EXTERNAL_MIN: float = -100.0
TEMP_EXTERNAL_MAX: float =  125.0

# Source: ESA Advanced Concepts Team (2021)
ENERGY_MIN_PCT: float = 60.0

# Source: NASA SBIR Thermal Management (SIMULATED order of magnitude)
PRESSURE_MIN_BAR: float = 20.0
PRESSURE_MAX_BAR: float = 40.0

# Minimum autonomy required for launch
AUTONOMY_MIN_HOURS: float = 2.0

# Decision strings — defined by the activity specification
DECISION_READY:   str = "PRONTO PARA DECOLAR"
DECISION_ABORTED: str = "DECOLAGEM ABORTADA"


# --- Pure functions — each returns bool, no side effects ---

def check_internal_temperature(value: float) -> bool:
    """Return True if internal electronics temperature is within operational range.

    Range: -20°C to +70°C
    Source: MIT OCW Satellite Engineering (2003) — digital electronics
    """
    return TEMP_INTERNAL_MIN <= value <= TEMP_INTERNAL_MAX


def check_external_temperature(value: float) -> bool:
    """Return True if external hull temperature is within survival range.

    Range: -100°C to +125°C
    Source: MIT OCW / ESA Bulletin 87 — structural panels
    """
    return TEMP_EXTERNAL_MIN <= value <= TEMP_EXTERNAL_MAX


def check_structural_integrity(flag: int) -> bool:
    """Return True if structural integrity flag indicates nominal state.

    Expected: 1 (nominal). 0 indicates detected structural failure.
    Source: ESA Mars Express dataset — binary status flag model
    """
    return flag == 1


def check_energy_level(pct: float) -> bool:
    """Return True if battery charge meets minimum launch threshold.

    Minimum: 60%
    Source: ESA Advanced Concepts Team (2021) — operational safety margin
    """
    return pct >= ENERGY_MIN_PCT


def check_tank_pressure(bar: float) -> bool:
    """Return True if propulsion tank pressure is within operational range.

    Range: 20 to 40 bar
    Source: NASA SBIR Thermal Management (SIMULATED)
    """
    return PRESSURE_MIN_BAR <= bar <= PRESSURE_MAX_BAR


def check_critical_modules(reading: TelemetryReading) -> bool:
    """Return True only if ALL critical subsystems are operational.

    Subsystems: propulsion, power management, communications,
                thermal control, navigation/ADCS.
    Source: ESA Mars Express ESOC — primary subsystem list
    """
    return all([
        reading.propulsion_ok,
        reading.power_ok,
        reading.comms_ok,
        reading.thermal_ok,
        reading.navigation_ok,
    ])


def check_autonomy(autonomy_hours: float) -> bool:
    """Return True if calculated energy autonomy meets minimum mission requirement.

    Minimum: 2.0 hours
    """
    return autonomy_hours >= AUTONOMY_MIN_HOURS


def run_checks(
    reading: TelemetryReading,
    autonomy_hours: float,
) -> Dict[str, bool]:
    """Return individual pass/fail result for each pre-launch check.

    Pure data — no formatting, no decision string.
    """
    return {
        "internal_temperature": check_internal_temperature(reading.internal_temperature),
        "external_temperature": check_external_temperature(reading.external_temperature),
        "structural_integrity": check_structural_integrity(reading.structural_integrity),
        "energy_level"        : check_energy_level(reading.energy_level_pct),
        "tank_pressure"       : check_tank_pressure(reading.tank_pressure_bar),
        "critical_modules"    : check_critical_modules(reading),
        "autonomy"            : check_autonomy(autonomy_hours),
    }


def decide_launch(checks: Dict[str, bool]) -> str:
    """Return DECISION_READY if all checks passed, DECISION_ABORTED otherwise."""
    return DECISION_READY if all(checks.values()) else DECISION_ABORTED


def verify_launch_readiness(
    reading: TelemetryReading,
    autonomy_hours: float,
) -> Tuple[str, Dict[str, bool]]:
    """Run all pre-launch checks and return final launch decision.

    Returns:
        Tuple of (decision_string, checks_dict)
        decision_string: DECISION_READY | DECISION_ABORTED
        checks_dict: individual pass/fail result per check
    """
    checks = run_checks(reading, autonomy_hours)
    return decide_launch(checks), checks


CHECK_LABELS: Dict[str, str] = {
    "internal_temperature": "Temperatura interna",
    "external_temperature": "Temperatura externa",
    "structural_integrity": "Integridade estrutural",
    "energy_level":         "Nível de energia",
    "tank_pressure":        "Pressão dos tanques",
    "critical_modules":     "Módulos críticos",
    "autonomy":             "Autonomia",
}


def print_checklist(decision: str, checks: Dict[str, bool]) -> None:
    """Print formatted verification checklist. Display-only, no return value."""
    print(SEPARATOR)
    print("🔍 CHECKLIST DE PRÉ-DECOLAGEM")
    print(SEPARATOR)
    for name, passed in checks.items():
        icon   = "✔︎" if passed else "💥"
        result = "OK" if passed else "FALHA"
        label  = CHECK_LABELS.get(name, name.replace("_", " ").title())
        print(f"  {icon}  {label:<30} {result}")
    print(SEPARATOR)
    print(f"\n  >>> {decision} <<<")
    if decision == DECISION_READY:
        print("  Pré-decolagem concluída. Tripulação, a seus postos. Vamos voar! 🚀")
    else:
        print("  Anomalia detectada. Os dados não mentem. Suspender a decolagem. 💥")
    print()
    print(SEPARATOR)

---
## ⚡ Seção 3 — Análise Energética

In [5]:
# =============================================================================
# SECTION 3 — ENERGY ANALYSIS
# =============================================================================
# Formulas from Cap. 7 FIAP — Energy Efficiency Fundamentals:
#   η  = E_util / E_total                    (system efficiency)
#   P  = I² × R                              (resistive thermal losses)
#   FC = avg_demand / peak_demand             (load factor)
# =============================================================================


@dataclass(frozen=True)
class EnergyAnalysis:
    """Immutable result of the pre-launch energy calculation."""
    total_capacity_kwh:   float
    charge_pct:           float
    current_charge_kwh:   float
    energy_losses_kwh:    float
    efficiency:           float
    energy_available_kwh: float
    consumption_kw:       float
    autonomy_hours:       float
    load_factor:          float


def compute_energy_analysis(
    total_capacity_kwh: float,
    charge_pct: float,
    consumption_kw: float,
    efficiency: float,
    loss_factor: float,
) -> EnergyAnalysis:
    """Compute full energy analysis for pre-launch verification.

    Returns EnergyAnalysis with all derived metrics (Cap. 7 FIAP formulas).
    """
    # Cap. 7 FIAP — current_charge = capacity × (charge_pct / 100)
    current_charge_kwh = total_capacity_kwh * (charge_pct / 100)

    # Cap. 7 FIAP — P = I²×R (resistive thermal dissipation)
    energy_losses_kwh = current_charge_kwh * loss_factor

    # Cap. 7 FIAP — η = E_util / E_total
    energy_available_kwh = current_charge_kwh * efficiency

    # Autonomy = available energy / launch consumption
    autonomy_hours = energy_available_kwh / consumption_kw

    # Cap. 7 FIAP — FC = avg_demand / peak_demand
    load_factor = current_charge_kwh / total_capacity_kwh

    return EnergyAnalysis(
        total_capacity_kwh=total_capacity_kwh,
        charge_pct=charge_pct,
        current_charge_kwh=current_charge_kwh,
        energy_losses_kwh=energy_losses_kwh,
        efficiency=efficiency,
        energy_available_kwh=energy_available_kwh,
        consumption_kw=consumption_kw,
        autonomy_hours=autonomy_hours,
        load_factor=load_factor,
    )


def print_energy_analysis(ea: EnergyAnalysis) -> None:
    """Print formatted energy analysis report. Display-only, no return value."""
    autonomy_ok = ea.autonomy_hours >= AUTONOMY_MIN_HOURS
    print(SEPARATOR)
    print("⚡ ANÁLISE ENERGÉTICA — AURORA SIGER")
    print(SEPARATOR)
    print(f"  Capacidade total       : {ea.total_capacity_kwh:>8.1f} kWh")
    print(f"  Carga atual            : {ea.charge_pct:>8.1f}%  →  {ea.current_charge_kwh:.2f} kWh")
    print(f"  Perdas resistivas (I²R): {ea.energy_losses_kwh:>8.2f} kWh  ({LOSS_FACTOR:.0%} fator de perda)")
    print(f"  Rendimento η           : {ea.efficiency:>8.0%}")
    print(f"  Energia disponível     : {ea.energy_available_kwh:>8.2f} kWh")
    print(f"  Consumo no lançamento  : {ea.consumption_kw:>8.1f} kW")
    print(f"  Autonomia              : {ea.autonomy_hours:>8.2f} h")
    print(f"  Fator de carga (FC)    : {ea.load_factor:>8.2f}")
    print(SEPARATOR)
    print(f"  Mínimo exigido         : {AUTONOMY_MIN_HOURS:>8.1f} h")
    print(f"  Status                 : {format_status(autonomy_ok)}")
    print(SEPARATOR)


# --- Compute energy and run full verification ---
energy = compute_energy_analysis(
    total_capacity_kwh=TOTAL_CAPACITY_KWH,
    charge_pct=TELEMETRY.energy_level_pct,
    consumption_kw=LAUNCH_CONSUMPTION_KW,
    efficiency=SYSTEM_EFFICIENCY,
    loss_factor=LOSS_FACTOR,
)

decision, checks = verify_launch_readiness(TELEMETRY, energy.autonomy_hours)

print_energy_analysis(energy)
print()
print_checklist(decision, checks)

⚡ ANÁLISE ENERGÉTICA — AURORA SIGER
  Capacidade total       :    120.0 kWh
  Carga atual            :     87.3%  →  104.76 kWh
  Perdas resistivas (I²R):     8.38 kWh  (8% fator de perda)
  Rendimento η           :      92%
  Energia disponível     :    96.38 kWh
  Consumo no lançamento  :      8.5 kW
  Autonomia              :    11.34 h
  Fator de carga (FC)    :     0.87
  Mínimo exigido         :      2.0 h
  Status                 : OK

🔍 CHECKLIST DE PRÉ-DECOLAGEM
  ✔︎  Temperatura interna            OK
  ✔︎  Temperatura externa            OK
  ✔︎  Integridade estrutural         OK
  ✔︎  Nível de energia               OK
  ✔︎  Pressão dos tanques            OK
  ✔︎  Módulos críticos               OK
  ✔︎  Autonomia                      OK

  >>> PRONTO PARA DECOLAR <<<
  Pré-decolagem concluída. Tripulação, a seus postos. Vamos voar! 🚀



---
## 👾 Seção 4 — Análise Assistida por IA

Dois algoritmos complementares:

1. **IsolationForest** (não supervisionado) — detecta desvios do padrão nominal sem precisar de rótulos históricos.
2. **DecisionTreeClassifier** (supervisionado) — aprende regras if-then a partir de histórico rotulado; geometricamente compatível com o padrão de anomalias (OR de condições por feature).

Dataset sintético gerado a partir das constantes da Seção 2 — consistência garantida entre verificação e treinamento.

> ⚠️ **Recall é a métrica prioritária**: falso negativo é mais perigoso que falso positivo em sistemas de segurança crítica.

In [6]:
# =============================================================================
# SECTION 4 — AI-ASSISTED ANALYSIS
# =============================================================================
# Dataset: synthetic, generated from the safety thresholds defined in Section 2
# Sources: NASA SMAP/MSL (Hundman et al., KDD 2018); MIT OCW; ESA MEX
# =============================================================================

# Feature column order — shared by both dataset generators and classifiers
FEATURE_NAMES: List[str] = [
    "internal_temp",
    "external_temp",
    "structural_integrity",
    "energy_pct",
    "tank_pressure",
]

# Nominal ranges — within safe limits, representing typical operation.
# Derived from Section 2 constants — single source of truth.
NOMINAL_RANGES: Dict[str, Tuple[float, float]] = {
    "internal_temp"       : (TEMP_INTERNAL_MIN + 5,  TEMP_INTERNAL_MAX - 5),
    "external_temp"       : (TEMP_EXTERNAL_MIN + 10, TEMP_EXTERNAL_MAX - 15),
    "structural_integrity": (1.0, 1.0),
    "energy_pct"          : (ENERGY_MIN_PCT + 2,     100.0),
    "tank_pressure"       : (PRESSURE_MIN_BAR + 2,   PRESSURE_MAX_BAR - 2),
}

# Anomaly ranges — deliberately outside safe limits.
# Each feature maps to one or more out-of-range bands (lo, hi).
# Inspired by NASA SMAP/MSL anomaly classes (Hundman et al., KDD 2018).
ANOMALY_RANGES: Dict[str, List[Tuple[float, float]]] = {
    "internal_temp"       : [(TEMP_INTERNAL_MIN - 30, TEMP_INTERNAL_MIN - 1),
                             (TEMP_INTERNAL_MAX + 1,  TEMP_INTERNAL_MAX + 40)],
    "external_temp"       : [(TEMP_EXTERNAL_MIN - 80, TEMP_EXTERNAL_MIN - 1),
                             (TEMP_EXTERNAL_MAX + 1,  TEMP_EXTERNAL_MAX + 75)],
    "structural_integrity": [(0.0, 0.0)],
    "energy_pct"          : [(5.0, ENERGY_MIN_PCT - 1)],
    "tank_pressure"       : [(0.0,                 PRESSURE_MIN_BAR - 1),
                             (PRESSURE_MAX_BAR + 1, PRESSURE_MAX_BAR + 40)],
}


# --- Pure dataset generation functions ---

def _sample_uniform(lo: float, hi: float) -> float:
    """Sample a single float uniformly from [lo, hi]."""
    return float(np.random.uniform(lo, hi))


def _sample_nominal(feature: str) -> float:
    """Return one nominal (safe) value for the given feature."""
    lo, hi = NOMINAL_RANGES[feature]
    return _sample_uniform(lo, hi)


def _sample_anomaly(feature: str) -> float:
    """Return one anomalous value for the given feature.

    Picks a random out-of-range band, then samples from it.
    """
    bands   = ANOMALY_RANGES[feature]
    lo, hi  = bands[np.random.randint(len(bands))]
    return _sample_uniform(lo, hi)


def _inject_fault(row: np.ndarray, fault_feature: str) -> np.ndarray:
    """Return a copy of row with one feature replaced by an anomalous value."""
    col    = FEATURE_NAMES.index(fault_feature)
    result = row.copy()
    result[col] = _sample_anomaly(fault_feature)
    return result


def telemetry_to_feature_array(reading: TelemetryReading) -> np.ndarray:
    """Convert a TelemetryReading into the feature array expected by the models."""
    return np.array([[
        reading.internal_temperature,
        reading.external_temperature,
        reading.structural_integrity,
        reading.energy_level_pct,
        reading.tank_pressure_bar,
    ]])


def generate_nominal_missions(n: int) -> np.ndarray:
    """Generate n rows of nominal telemetry within safe operational ranges."""
    return np.array(
        [[_sample_nominal(f) for f in FEATURE_NAMES] for _ in range(n)]
    )


def generate_anomalous_missions(n: int) -> np.ndarray:
    """Generate n rows of telemetry each with exactly one injected fault.

    Fault feature is chosen uniformly at random per row.
    Source: NASA SMAP/MSL anomaly structure — Hundman et al., KDD 2018.
    """
    fault_features = list(ANOMALY_RANGES.keys())
    chosen_faults  = np.random.choice(fault_features, size=n)
    nominal_rows   = generate_nominal_missions(n)

    return np.array([
        _inject_fault(row, fault)
        for row, fault in zip(nominal_rows, chosen_faults)
    ])


# Dataset label constants
LABEL_NOMINAL: int = 0   # mission within all safe operational limits
LABEL_ANOMALY: int = 1   # mission with at least one parameter out of range


def build_dataset(
    n_nominal: int,
    n_anomalous: int,
) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Build labeled feature matrix X, label vector y, and summary DataFrame."""
    X_nominal   = generate_nominal_missions(n_nominal)
    X_anomalous = generate_anomalous_missions(n_anomalous)

    X = np.vstack([X_nominal, X_anomalous])
    y = np.array([LABEL_NOMINAL] * n_nominal + [LABEL_ANOMALY] * n_anomalous)

    df = pd.DataFrame(X, columns=FEATURE_NAMES)
    df["label"]      = y
    df["label_name"] = df["label"].map({LABEL_NOMINAL: "nominal", LABEL_ANOMALY: "anomaly"})

    return X, y, df


# --- Build dataset ---
# 2000 missions: 1500 nominal (75%) + 500 anomalous (25%).
# With 5-fold CV: each test fold has ~100 anomalies, keeping recall std below 3%.
N_NOMINAL:   int = 1500
N_ANOMALOUS: int = 500

X, y, df = build_dataset(N_NOMINAL, N_ANOMALOUS)

print(f"🌌 Dataset gerado: {N_NOMINAL} nominais + {N_ANOMALOUS} anômalas = {len(df)} missões")
print(df.groupby("label_name")[FEATURE_NAMES].mean().round(2))

🌌 Dataset gerado: 1500 nominais + 500 anômalas = 2000 missões
            internal_temp  external_temp  structural_integrity  energy_pct  \
label_name                                                                   
anomaly             27.02           4.98                   0.8       72.31   
nominal             24.25           9.45                   1.0       81.10   

            tank_pressure  
label_name                 
anomaly             31.02  
nominal             29.94  


In [7]:
# =============================================================================
# 4A — ANOMALY DETECTION: IsolationForest (unsupervised)
# =============================================================================

# IsolationForest output convention — documented here for remapping
IF_ANOMALY_LABEL: int = -1   # IsolationForest marks anomalies as -1
IF_INLIER_LABEL:  int = +1   # IsolationForest marks nominal as +1

CONTAMINATION_RATIO: float = N_ANOMALOUS / (N_NOMINAL + N_ANOMALOUS)


@dataclass(frozen=True)
class IsolationForestResult:
    """Immutable result of IsolationForest training and classification."""
    scaler:    StandardScaler
    model:     IsolationForest
    labels:    np.ndarray     # remapped: LABEL_NOMINAL=0, LABEL_ANOMALY=1
    recall:    float
    precision: float
    f1:        float
    cm:        np.ndarray


def train_isolation_forest(
    X: np.ndarray,
    y: np.ndarray,
    contamination: float,
) -> IsolationForestResult:
    """Fit an IsolationForest and evaluate against ground-truth labels.

    Returns an immutable result object — no global state modified.
    """
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    model  = IsolationForest(contamination=contamination, random_state=RANDOM_STATE)
    preds  = model.fit_predict(X_scaled)

    # Remap IsolationForest output to project label convention
    labels = (preds == IF_ANOMALY_LABEL).astype(int)

    return IsolationForestResult(
        scaler=scaler,
        model=model,
        labels=labels,
        recall=recall_score(y, labels),
        precision=precision_score(y, labels),
        f1=f1_score(y, labels),
        cm=confusion_matrix(y, labels),
    )


def classify_with_isolation_forest(
    result: IsolationForestResult,
    reading: TelemetryReading,
) -> str:
    """Classify a single telemetry reading. Returns 'ANOMALIA DETECTADA' or 'NOMINAL'."""
    features = telemetry_to_feature_array(reading)
    scaled   = result.scaler.transform(features)
    pred     = result.model.predict(scaled)[0]
    return "ANOMALIA DETECTADA" if pred == IF_ANOMALY_LABEL else "NOMINAL"


def print_isolation_forest_report(
    result: IsolationForestResult,
    mission_classification: str,
) -> None:
    """Print IsolationForest evaluation report. Display-only."""
    print(SEPARATOR)
    print("🦉 IsolationForest — Detecção de Anomalias (não supervisionado)")
    print(SEPARATOR)
    print(f"  Recall    : {result.recall:.2%}   ← métrica prioritária em sistemas de segurança crítica")
    print(f"  Precisão  : {result.precision:.2%}")
    print(f"  F1        : {result.f1:.2%}")
    print()
    print("  Matriz de confusão (linhas=real, colunas=previsto):")
    print("                   Nominal  Anomalia")
    print(f"  Real Nominal   : {result.cm[0][0]:>6}   {result.cm[0][1]:>6}")
    print(f"  Real Anomalia  : {result.cm[1][0]:>6}   {result.cm[1][1]:>6}")
    print(SEPARATOR)
    print(f"  Missão Aurora Siger : {mission_classification}")


iso        = train_isolation_forest(X, y, CONTAMINATION_RATIO)
iso_result = classify_with_isolation_forest(iso, TELEMETRY)
print_isolation_forest_report(iso, iso_result)

🦉 IsolationForest — Detecção de Anomalias (não supervisionado)
  Recall    : 94.80%   ← métrica prioritária em sistemas de segurança crítica
  Precisão  : 94.80%
  F1        : 94.80%

  Matriz de confusão (linhas=real, colunas=previsto):
                   Nominal  Anomalia
  Real Nominal   :   1474       26
  Real Anomalia  :     26      474
  Missão Aurora Siger : NOMINAL


In [8]:
# =============================================================================
# 4B — BINARY CLASSIFICATION: DecisionTreeClassifier (supervised)
# =============================================================================
# Decision trees learn if-then rules — same geometry as the Section 2 checklist.
# Trained here, the tree autonomously rediscovers the safety thresholds.
# Scale-invariant: does not require feature normalization.
#
# Setup: class_weight='balanced' (3:1 imbalance), StratifiedKFold CV (no data leakage).
# =============================================================================

TEST_SIZE:    float = 0.25   # fraction of data held out for hold-out evaluation
CV_FOLDS:     int   = 5      # number of stratified folds for cross-validation
DT_MAX_DEPTH: int   = 10     # upper bound on tree depth; actual depth may be lower
TREE_DISPLAY_DEPTH: int = 3  # levels shown in the printed rule summary


@dataclass(frozen=True)
class DecisionTreeResult:
    """Immutable result of DecisionTreeClassifier training and classification."""
    model:          DecisionTreeClassifier
    y_test:         np.ndarray
    y_pred:         np.ndarray
    recall:         float
    precision:      float
    f1:             float
    mission_label:  int
    mission_proba:  np.ndarray
    cv_recall_mean: float   # mean recall across CV_FOLDS stratified folds
    cv_recall_std:  float   # standard deviation — lower means more stable estimate
    tree_depth:     int     # actual depth learned by the model
    n_leaves:       int     # number of leaf nodes (model complexity indicator)


def train_decision_tree(
    X: np.ndarray,
    y: np.ndarray,
    reading: TelemetryReading,
    test_size: float,
    cv_folds: int,
) -> DecisionTreeResult:
    """Train a DecisionTreeClassifier and classify a single telemetry reading.

    Returns an immutable result object — no global state modified.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
    )

    model = DecisionTreeClassifier(
        max_depth=DT_MAX_DEPTH,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Cross-validation — no Pipeline needed: DecisionTree is scale-invariant
    kfold     = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(
        DecisionTreeClassifier(
            max_depth=DT_MAX_DEPTH,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        X, y, cv=kfold, scoring="recall",
    )

    features      = telemetry_to_feature_array(reading)
    mission_label = model.predict(features)[0]
    mission_proba = model.predict_proba(features)[0]

    return DecisionTreeResult(
        model=model,
        y_test=y_test,
        y_pred=y_pred,
        recall=recall_score(y_test, y_pred),
        precision=precision_score(y_test, y_pred),
        f1=f1_score(y_test, y_pred),
        mission_label=int(mission_label),
        mission_proba=mission_proba,
        cv_recall_mean=float(cv_scores.mean()),
        cv_recall_std=float(cv_scores.std()),
        tree_depth=int(model.get_depth()),
        n_leaves=int(model.get_n_leaves()),
    )


def print_decision_tree_report(
    result: DecisionTreeResult,
    feature_names: List[str],
) -> None:
    """Print DecisionTree evaluation report including learned rules. Display-only."""
    dt_classification = "ANOMALIA DETECTADA" if result.mission_label == LABEL_ANOMALY else "NOMINAL"

    print(SEPARATOR)
    print("🌲 DecisionTreeClassifier — Classificação Binária (supervisionado)")
    print(SEPARATOR)
    print(f"  Recall divisão única (anomalia): {result.recall:.2%}")
    print(f"  Recall CV {CV_FOLDS}-fold médio      : {result.cv_recall_mean:.2%} ± {result.cv_recall_std:.2%}   ← estimativa mais robusta")
    print(f"  Precisão (anomalia)         : {result.precision:.2%}")
    print(f"  F1 (anomalia)               : {result.f1:.2%}")
    print(f"  Profundidade da árvore      : {result.tree_depth}")
    print(f"  Folhas                      : {result.n_leaves}")
    print(SEPARATOR)
    print(f"  Missão Aurora Siger  : {dt_classification}")
    print(f"  Probabilidade nominal : {result.mission_proba[0]:.2%}")
    print(f"  Probabilidade anomalia: {result.mission_proba[1]:.2%}")
    print()
    print(f"  Regras aprendidas pela árvore (primeiros {TREE_DISPLAY_DEPTH} níveis):")
    print(export_text(result.model, feature_names=feature_names, max_depth=TREE_DISPLAY_DEPTH))


dt_result = train_decision_tree(X, y, TELEMETRY, TEST_SIZE, CV_FOLDS)
print_decision_tree_report(dt_result, FEATURE_NAMES)

🌲 DecisionTreeClassifier — Classificação Binária (supervisionado)
  Recall divisão única (anomalia): 100.00%
  Recall CV 5-fold médio      : 99.80% ± 0.40%   ← estimativa mais robusta
  Precisão (anomalia)         : 100.00%
  F1 (anomalia)               : 100.00%
  Profundidade da árvore      : 8
  Folhas                      : 12
  Missão Aurora Siger  : NOMINAL
  Probabilidade nominal : 100.00%
  Probabilidade anomalia: 0.00%

  Regras aprendidas pela árvore (primeiros 3 níveis):
|--- structural_integrity <= 0.50
|   |--- class: 1
|--- structural_integrity >  0.50
|   |--- energy_pct <= 62.21
|   |   |--- energy_pct <= 62.01
|   |   |   |--- tank_pressure <= 22.65
|   |   |   |   |--- class: 1
|   |   |   |--- tank_pressure >  22.65
|   |   |   |   |--- class: 1
|   |   |--- energy_pct >  62.01
|   |   |   |--- internal_temp <= -23.23
|   |   |   |   |--- class: 1
|   |   |   |--- internal_temp >  -23.23
|   |   |   |   |--- class: 0
|   |--- energy_pct >  62.21
|   |   |--- internal

In [9]:
# =============================================================================
# 4C — AI RISK SUMMARY
# Covers: data classification, anomaly identification, risk suggestion
# =============================================================================

# Risk level thresholds
RISK_LOW_THRESHOLD:      float = 0.20  # below 20% anomaly probability → BAIXO
RISK_MODERATE_THRESHOLD: float = 0.50  # below 50% → MODERADO; above → ALTO


def classify_risk_level(anomaly_probability: float) -> str:
    """Return risk level string based on anomaly probability."""
    if anomaly_probability < RISK_LOW_THRESHOLD:
        return "BAIXO"
    if anomaly_probability < RISK_MODERATE_THRESHOLD:
        return "MODERADO"
    return "ALTO"


def build_ai_risk_summary(
    isolation_forest_result: str,
    isolation_forest_recall: float,
    dt_result: DecisionTreeResult,
) -> str:
    """Build and return the AI risk assessment string.

    Pure function — no side effects.
    """
    dt_classification = "ANOMALIA DETECTADA" if dt_result.mission_label == LABEL_ANOMALY else "NOMINAL"
    anomaly_prob      = dt_result.mission_proba[1]
    risk_level        = classify_risk_level(anomaly_prob)
    consensus         = isolation_forest_result == dt_classification
    recommendation    = "PROSSEGUIR COM CAUTELA" if risk_level == "BAIXO" else "AGUARDAR E REVISAR"

    lines = [
        SEPARATOR,
        "☄️  AVALIAÇÃO DE RISCO — AURORA SIGER",
        SEPARATOR,
        "  Classificação dos dados",
        f"    IsolationForest (não supervisionado) : {isolation_forest_result}",
        f"    DecisionTree    (supervisionado)   : {dt_classification}",
        f"    Consenso entre modelos         : {'SIM' if consensus else 'NÃO — REVISÃO NECESSÁRIA'}",
        "",
        "  Identificação de anomalia",
        f"    Probabilidade de anomalia     : {anomaly_prob:.1%}",
        f"    Recall IsolationForest        : {isolation_forest_recall:.1%}",
        f"    Recall DecisionTree (CV {CV_FOLDS}-fold): {dt_result.cv_recall_mean:.1%} ± {dt_result.cv_recall_std:.1%}",
        "",
        "  Avaliação de risco",
        f"    Nível de risco               : {risk_level}",
        f"    Ação recomendada             : {recommendation}",
        "",
        SEPARATOR,
        "  NOTA: A análise de IA é apenas suporte. A decisão final",
        "  cabe ao operador humano. Sistemas automatizados não devem",
        "  sobrepor o juízo humano em contextos de segurança crítica.",
        SEPARATOR,
    ]
    return "\n".join(lines)


print(build_ai_risk_summary(iso_result, iso.recall, dt_result))

☄️  AVALIAÇÃO DE RISCO — AURORA SIGER
  Classificação dos dados
    IsolationForest (não supervisionado) : NOMINAL
    DecisionTree    (supervisionado)   : NOMINAL
    Consenso entre modelos         : SIM

  Identificação de anomalia
    Probabilidade de anomalia     : 0.0%
    Recall IsolationForest        : 94.8%
    Recall DecisionTree (CV 5-fold): 99.8% ± 0.4%

  Avaliação de risco
    Nível de risco               : BAIXO
    Ação recomendada             : PROSSEGUIR COM CAUTELA

  NOTA: A análise de IA é apenas suporte. A decisão final
  cabe ao operador humano. Sistemas automatizados não devem
  sobrepor o juízo humano em contextos de segurança crítica.


In [10]:
# =============================================================================
# FINAL OUTPUT — COMPLETE MISSION REPORT SUMMARY
# =============================================================================

def print_final_report(
    ea: EnergyAnalysis,
    iso_result: str,
    dt_result: DecisionTreeResult,
    decision: str,
) -> None:
    """Print the complete mission pre-launch report summary. Display-only."""
    dt_classification = "ANOMALIA DETECTADA" if dt_result.mission_label == LABEL_ANOMALY else "NOMINAL"

    print("\n" + SEPARATOR)
    print("  ☄️  AURORA SIGER — RELATÓRIO FINAL DE MISSÃO")
    print(SEPARATOR)
    print(f"  Energia disponível   : {ea.energy_available_kwh:.2f} kWh")
    print(f"  Autonomia            : {ea.autonomy_hours:.2f} h")
    print(f"  IsolationForest      : {iso_result}")
    print(f"  DecisionTree         : {dt_classification}")
    print(f"  Prob. de anomalia    : {dt_result.mission_proba[1]:.1%}")
    print()
    print(SEPARATOR)
    print(f"  >>> {decision} <<<")
    if decision == DECISION_READY:
        print("  O universo espera. Boa viagem, Aurora Siger. Vida longa e próspera. 🖖")
    else:
        print("  Resistência é inútil. Os parâmetros exigem correção. Missão suspensa. 💀")
    print(SEPARATOR)


print_final_report(energy, iso_result, dt_result, decision)


  ☄️  AURORA SIGER — RELATÓRIO FINAL DE MISSÃO
  Energia disponível   : 96.38 kWh
  Autonomia            : 11.34 h
  IsolationForest      : NOMINAL
  DecisionTree         : NOMINAL
  Prob. de anomalia    : 0.0%

  >>> PRONTO PARA DECOLAR <<<
  O universo espera. Boa viagem, Aurora Siger. Vida longa e próspera. 🖖
